# Study dataset column reduction

Both study datasets have too many columns for a time-boxed EDA session — participants spend
their budget reading the schema instead of exploring. This notebook produces reduced versions
of both datasets that cut mechanical repetition while preserving every *conceptual* dimension
participants could explore.

| dataset | raw | reduced | what gets cut |
|---|---|---|---|
| FAA Wildlife Strikes (`faa_damage.csv`) | 61 cols | 21 cols | 32 binary per-component strike/damage flags (collapsed into 3 derived severity measures), redundant IDs, near-empty fields |
| Health Spending (`health_spending.csv`) | 57 cols | 23 cols | only the 34 confidence-interval bound columns; all mean measures, `location_id`, `location_name`, `level`, and all rows are kept, with measures renamed to plain English |

**This notebook does NOT touch the original CSVs.** It writes `faa_damage_reduced.csv` and
`health_spending_reduced.csv` next to them in `example_datasets/`. The app only surfaces files
explicitly listed in `example_datasets_config.py`, so the reduced files stay invisible until we
swap the config paths.

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path.cwd().parent / "example_datasets"
assert DATA_DIR.exists(), f"expected {DATA_DIR} to exist — run from example_analysis/"

# --- decision still open; flip to change the plan ---
# Keep LATITUDE/LONGITUDE in the FAA data? They enable map scatters but are 79% filled
# and mostly duplicate AIRPORT/STATE. Default: drop.
KEEP_FAA_LATLON = False

faa_raw = pd.read_csv(DATA_DIR / "faa_damage.csv", low_memory=False)
health_raw = pd.read_csv(DATA_DIR / "health_spending.csv")
print(f"FAA raw:    {faa_raw.shape[0]:>6} rows x {faa_raw.shape[1]} cols")
print(f"Health raw: {health_raw.shape[0]:>6} rows x {health_raw.shape[1]} cols")

FAA raw:     21948 rows x 61 cols
Health raw:   6048 rows x 57 cols


## 1. FAA Wildlife Strikes: 61 → 21 columns

The reduction keeps every analysis dimension — *when* (date, time of day), *where* (airport,
state), *who* (operator), *what aircraft* (type, class, mass, engines), *flight context*
(phase, height, speed, sky), *what animal* (species, size, number struck), and *outcome*
(damage level) — and collapses only the wall of 32 binary component flags:

- **Derived, replacing all 32 `STR_*` / `DAM_*` / `ING_*` flags:**
  - `PARTS_STRUCK` — how many of the 14 aircraft components were struck
  - `PARTS_DAMAGED` — how many were damaged
  - `ENGINE_INGESTION` — whether any engine ingested the animal (Yes/No)

  This keeps the "how bad was it" dimension explorable (it correlates with `DAMAGE_LEVEL`,
  `SIZE`, `PHASE_OF_FLIGHT`) without 32 near-meaningless 0/1 columns.
- **Dropped as redundant:** `INDEX_NR` (record id), `SPECIES_ID` (≈`SPECIES`), `OPID` (≈`OPERATOR`)
- **Dropped as too sparse to explore:** `PRECIPITATION` (6% filled), `DISTANCE` (47%),
  `ENG_1_POS`…`ENG_4_POS` (positions 3/4 are 6% and 2.5% filled; 1/2 are aircraft trivia)
- **Lat/long:** dropped by default (`KEEP_FAA_LATLON`), since `AIRPORT`/`STATE` already cover geography

In [2]:
# Sparsity of the columns slated for dropping (the flags are dense but binary-noise;
# these are the genuinely sparse ones).
sparse_candidates = ["PRECIPITATION", "DISTANCE", "SPEED", "HEIGHT",
                     "ENG_1_POS", "ENG_2_POS", "ENG_3_POS", "ENG_4_POS",
                     "LATITUDE", "LONGITUDE", "SKY", "TIME_OF_DAY"]
faa_raw[sparse_candidates].notna().mean().sort_values().rename("share filled").to_frame().style.format("{:.1%}")

,share filled
ENG_4_POS,2.5%
ENG_3_POS,5.9%
PRECIPITATION,6.3%
DISTANCE,47.1%
SPEED,55.7%
SKY,69.6%
HEIGHT,75.3%
ENG_2_POS,79.2%
LATITUDE,79.4%
LONGITUDE,79.4%


In [3]:
STR_FLAGS = [c for c in faa_raw.columns if c.startswith("STR_")]
DAM_FLAGS = [c for c in faa_raw.columns if c.startswith("DAM_")]
ING_FLAGS = [c for c in faa_raw.columns if c.startswith("ING_")]
print(f"{len(STR_FLAGS)} STR flags, {len(DAM_FLAGS)} DAM flags, {len(ING_FLAGS)} ING flags")

faa = faa_raw.copy()
faa["PARTS_STRUCK"] = faa[STR_FLAGS].sum(axis=1).astype(int)
faa["PARTS_DAMAGED"] = faa[DAM_FLAGS].sum(axis=1).astype(int)
faa["ENGINE_INGESTION"] = faa[ING_FLAGS].any(axis=1).map({True: "Yes", False: "No"})

FAA_KEEP = [
    # when
    "INCIDENT_DATE", "TIME_OF_DAY",
    # where
    "AIRPORT", "STATE",
    *(["LATITUDE", "LONGITUDE"] if KEEP_FAA_LATLON else []),
    # who / what aircraft
    "OPERATOR", "AIRCRAFT", "AC_CLASS", "AC_MASS", "TYPE_ENG", "NUM_ENGS",
    # flight context
    "PHASE_OF_FLIGHT", "HEIGHT", "SPEED", "SKY",
    # outcome
    "DAMAGE_LEVEL", "PARTS_STRUCK", "PARTS_DAMAGED", "ENGINE_INGESTION",
    # wildlife
    "SPECIES", "SIZE", "NUM_STRUCK",
]
faa_reduced = faa[FAA_KEEP]
print(f"FAA reduced: {faa_reduced.shape[0]} rows x {faa_reduced.shape[1]} cols")
faa_reduced.head()

14 STR flags, 14 DAM flags, 4 ING flags
FAA reduced: 21948 rows x 21 cols


,INCIDENT_DATE,TIME_OF_DAY,AIRPORT,STATE,OPERATOR,AIRCRAFT,AC_CLASS,AC_MASS,TYPE_ENG,NUM_ENGS,...,HEIGHT,SPEED,SKY,DAMAGE_LEVEL,PARTS_STRUCK,PARTS_DAMAGED,ENGINE_INGESTION,SPECIES,SIZE,NUM_STRUCK
0,1996-07-01,NaN,LA GUARDIA ARPT,NY,UNITED AIRLINES,A-320,airplane,"27,001-272,000 kg",turbofan,2.0,...,5000.0,NaN,NaN,Minor,2,1,No,Unknown bird - medium,Medium,1
1,1990-08-07,Night,LAMBERT-ST LOUIS INTL,MO,PRIVATELY OWNED,C-152,airplane,"2,250 kg or less",reciprocating engine,1.0,...,200.0,70.0,No Cloud,Minor,1,1,No,Unknown bird - large,Large,1
2,1995-04-28,NaN,SAN FRANCISCO INTL ARPT,CA,UNITED AIRLINES,B-757-200,airplane,"27,001-272,000 kg",turbofan,2.0,...,0.0,140.0,NaN,Substantial,3,3,No,Unknown bird - medium,Medium,11-100
3,1993-09-19,Day,MANCHESTER AIRPORT,NH,PRIVATELY OWNED,BE-33,airplane,"2,250 kg or less",reciprocating engine,1.0,...,1800.0,150.0,No Cloud,Substantial,1,1,No,Unknown bird - large,Large,1
4,1992-05-24,Day,UNKNOWN,NaN,PRIVATELY OWNED,C-180,airplane,"2,250 kg or less",reciprocating engine,1.0,...,1000.0,120.0,No Cloud,Minor,1,1,No,Bald eagle,Large,1


In [4]:
# The derived severity measures should track the official damage rating — sanity check.
faa_reduced.groupby("DAMAGE_LEVEL")[["PARTS_STRUCK", "PARTS_DAMAGED"]].mean().round(2)

,PARTS_STRUCK,PARTS_DAMAGED
DAMAGE_LEVEL,,
Destroyed,1.71,4.70
Minor,1.43,1.12
Substantial,1.68,1.43
Undetermined,1.45,1.15


## 2. Health Spending: 57 → 23 columns

The raw file is a 5 × 4 × 3 grid: five spending sources (THE total, GHES government,
PPP prepaid private, OOP out-of-pocket, DAH development assistance) × four normalizations
(absolute, per-capita, share-of-total, per-GDP) × three statistics (mean / lower / upper).

- **Drop only the 34 `_lower`/`_upper` CI-bound columns** — pure noise for participants.
  All 19 mean measures are kept, as are `location_id`, `location_name`, and `level`
  (so Country, GBD Super Region, World Bank Income Group, and Global rows all stay).
- **Rename the measures to plain English** — `ghes_per_the_mean` → `govt_share_of_spending` —
  which attacks the confusion problem without dropping anything.

Note: because all `level` rows are kept, participants who aggregate across rows without
filtering on `level` will double-count (countries are also contained in the region /
income-group / global aggregates). Worth flagging in the study materials or task framing.

In [5]:
print(health_raw["level"].value_counts().to_string())

n_ci = sum(c.endswith(("_lower", "_upper")) for c in health_raw.columns)
print(f"\n{n_ci} CI-bound columns to drop")

HEALTH_RENAME = {
    # identity (kept as-is)
    "location_id": "location_id",
    "location_name": "location_name",
    "level": "level",
    "year": "year",
    # absolute spending (constant USD)
    "the_total_mean": "total_health_spending",
    "ghes_total_mean": "govt_spending",
    "ppp_total_mean": "prepaid_private_spending",
    "oop_total_mean": "out_of_pocket_spending",
    "dah_total_mean": "dev_assistance_spending",
    # per capita (constant USD)
    "the_per_cap_mean": "spending_per_capita",
    "ghes_per_cap_mean": "govt_spending_per_capita",
    "ppp_per_cap_mean": "prepaid_private_per_capita",
    "oop_per_cap_mean": "out_of_pocket_per_capita",
    "dah_per_cap_mean": "dev_assistance_per_capita",
    # composition (fraction of total health spending)
    "ghes_per_the_mean": "govt_share_of_spending",
    "ppp_per_the_mean": "prepaid_private_share_of_spending",
    "oop_per_the_mean": "out_of_pocket_share_of_spending",
    "dah_per_the_mean": "dev_assistance_share_of_spending",
    # economic burden (fraction of GDP)
    "the_per_gdp_mean": "spending_share_of_gdp",
    "ghes_per_gdp_mean": "govt_share_of_gdp",
    "ppp_per_gdp_mean": "prepaid_private_share_of_gdp",
    "oop_per_gdp_mean": "out_of_pocket_share_of_gdp",
    "dah_per_gdp_mean": "dev_assistance_share_of_gdp",
}
# every non-CI column must be accounted for — nothing else gets dropped
assert set(HEALTH_RENAME) == {c for c in health_raw.columns if not c.endswith(("_lower", "_upper"))}

health_reduced = health_raw[list(HEALTH_RENAME)].rename(columns=HEALTH_RENAME)
print(f"\nHealth reduced: {health_reduced.shape[0]} rows x {health_reduced.shape[1]} cols "
      f"({health_reduced['location_name'].nunique()} locations, "
      f"{health_reduced['year'].min()}-{health_reduced['year'].max()})")
health_reduced.head()

level
Country                     5712
GBD Super Regions            196
World Bank Income Groups     112
Global                        28

34 CI-bound columns to drop

Health reduced: 6048 rows x 23 cols (216 locations, 1995-2022)


,location_id,location_name,level,year,total_health_spending,govt_spending,prepaid_private_spending,out_of_pocket_spending,dev_assistance_spending,spending_per_capita,...,dev_assistance_per_capita,govt_share_of_spending,prepaid_private_share_of_spending,out_of_pocket_share_of_spending,dev_assistance_share_of_spending,spending_share_of_gdp,govt_share_of_gdp,prepaid_private_share_of_gdp,out_of_pocket_share_of_gdp,dev_assistance_share_of_gdp
0,1,Global,Global,1995,3939586332,2248343815,926217404,752889062,12136051,689,...,2,0.571,0.235,0.191,0.003,0.084,0.048,0.02,0.016,0.0
1,1,Global,Global,1996,4054867111,2302603900,955221699,785546542,11494970,700,...,2,0.568,0.236,0.194,0.003,0.084,0.048,0.02,0.016,0.0
2,1,Global,Global,1997,4204761608,2371938986,993190512,828032784,11599326,717,...,2,0.564,0.236,0.197,0.003,0.084,0.047,0.02,0.016,0.0
3,1,Global,Global,1998,4380588265,2458699708,1036342416,872865758,12680383,737,...,2,0.561,0.237,0.199,0.003,0.085,0.048,0.02,0.017,0.0
4,1,Global,Global,1999,4555473738,2543978336,1082760499,915256361,13478542,757,...,2,0.558,0.238,0.201,0.003,0.085,0.047,0.02,0.017,0.0


In [6]:
# Sanity check: the four source shares should roughly sum to 1 of total spending.
share_cols = [c for c in health_reduced.columns if c.endswith("share_of_spending")]
share_sum = health_reduced[share_cols].sum(axis=1)
print(f"share sum: median={share_sum.median():.3f}, 5th-95th pct="
      f"[{share_sum.quantile(0.05):.3f}, {share_sum.quantile(0.95):.3f}]")

share sum: median=1.000, 5th-95th pct=[0.999, 1.001]


## 3. Write reduced files

Written alongside the originals as `*_reduced.csv`; the originals are untouched.

In [7]:
for name, df in [("faa_damage_reduced.csv", faa_reduced),
                 ("health_spending_reduced.csv", health_reduced)]:
    out = DATA_DIR / name
    df.to_csv(out, index=False)
    print(f"wrote {out.relative_to(DATA_DIR.parent)}  ({df.shape[0]} rows x {df.shape[1]} cols)")

wrote example_datasets/faa_damage_reduced.csv  (21948 rows x 21 cols)
wrote example_datasets/health_spending_reduced.csv  (6048 rows x 23 cols)


## Next steps (deliberately not done here)

1. Review the reduced files (and the open FAA toggle: `KEEP_FAA_LATLON`).
2. Point `example_datasets_config.py` at the reduced files (or overwrite the originals) and
   update the two `description` strings to match the new schemas — the agents read those.
3. For the health data, decide how to handle the mixed `level` rows in study framing
   (aggregating across rows without filtering on `level` double-counts countries into the
   region / income-group / global aggregates).
4. Sanity-load both datasets in the app to confirm type inference.